# HANDS-ON PROJECT: HIERARCHICAL & AGGLOMERATIVE CLUSTERING
## Discovering wholesale customer purchasing patterns

**Course:** Data Analysis with Python / Business Analytics  
**Level:** Introductory–Intermediate | **Estimated work:** 4–6 hours | **Points:** 100  
**Dataset:** [Wholesale customers Data Set (Kaggle)](https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set)  
**Original source:** Cardoso (2013), [UCI Wholesale Customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers), CC BY 4.0.

**Business scenario.** You work as a junior data analyst for a wholesale distributor supplying restaurants, cafés, hotels, and other retail customers. Managers want to discover groups of customers with similar annual purchasing patterns. Your task is to investigate the hierarchy of customer similarities, select a reasonable segmentation, and develop *testable* business recommendations.

**Important:** This is **unsupervised clustering**, not prediction. The data do not identify customer profitability, response to promotions, or the effectiveness of marketing actions. Do not present any segment label as ground truth.

## Learning outcomes

After completing this project, you will be able to:
1. Inspect, validate, and visualize real wholesale spending data.
2. Explain how Euclidean distance and **single, complete, average, and Ward linkage** construct different cluster hierarchies.
3. Build and interpret a **dendrogram**, including the height at which two clusters merge.
4. Apply `AgglomerativeClustering` and select a defensible number of clusters.
5. Evaluate silhouette score, cluster balance, and stability; understand the limitations of each criterion.
6. Interpret clusters in original spending units and propose specific, cautious business actions.

**Workflow:** Data → EDA → Transformation → Pairwise distances → Linkage → Dendrogram → Tree cut → Validation → Business recommendations.

**Scoring (100 points):** Data quality 10; EDA 10; preprocessing 15; linkage theory 10; dendrograms 15; model selection 15; profiles and interpretation 15; sensitivity and recommendations 10.

## Part 0 — Environment and Kaggle dataset (not graded)

**Download options:**
- Open the [Kaggle dataset](https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set), download the CSV, and place **`Wholesale customers data.csv`** in the same folder as this notebook or in `data/`.
- Alternatively, use `kagglehub` to fetch the public Kaggle dataset automatically if an internet connection is available. If Kaggle authentication is required, follow the official instructions shown by Kaggle.

**The original CSV is not included in this teaching package.** Do not substitute arbitrary simulated data for your final analyses.

Packages: `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`, `kagglehub` (optional). Works in JupyterLab or Google Colab.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree, fcluster
from scipy.spatial.distance import cdist, pdist
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)

In [ ]:
# Load only the real original Kaggle file. You can also upload it in Colab.
candidates = [
    Path('Wholesale customers data.csv'),
    Path('data/Wholesale customers data.csv'),
    Path('Wholesale_customers_data.csv'),
    Path('data/Wholesale_customers_data.csv'),
]
found = next((p for p in candidates if p.is_file()), None)
if found is None:
    try:
        import kagglehub
        download_dir = Path(kagglehub.dataset_download('binovi/wholesale-customers-data-set'))
        all_csvs = sorted(download_dir.rglob('*.csv'))
        found = next((p for p in all_csvs if 'wholesale' in p.name.lower()), None)
        if found is None and len(all_csvs) == 1:
            found = all_csvs[0]
    except Exception as exc:
        print('Automatic Kaggle download unavailable:', str(exc)[:180])

if found is None:
    raise FileNotFoundError('Please download the original CSV from Kaggle and save it as Wholesale customers data.csv next to this notebook or in data/.')
df = pd.read_csv(found)
df.columns = df.columns.str.strip()
required_columns = ['Channel', 'Region', *spend_cols]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f'Unexpected CSV schema. Missing columns: {missing_columns}')
print('Loaded:', found, '| rows:', len(df), '| columns:', len(df.columns))
display(df.head())

### Data dictionary
| Column | Description | Role in clustering |
|---|---|---|
| `Fresh` | Annual purchases of fresh products, monetary units | Numerical feature |
| `Milk` | Annual purchases of milk products | Numerical feature |
| `Grocery` | Annual purchases of grocery products | Numerical feature |
| `Frozen` | Annual purchases of frozen products | Numerical feature |
| `Detergents_Paper` | Annual purchases of detergents/paper products | Numerical feature |
| `Delicassen` | Annual purchases of delicatessen products | Numerical feature |
| `Channel` | Customer channel (category stored as an integer) | **Post-clustering descriptive variable** |
| `Region` | Customer geographic region (category stored as an integer) | **Post-clustering descriptive variable** |

**Unit:** The source uses generic *monetary units (m.u.)*, not a specified national currency. Each record represents **one wholesale customer**, not a transaction. `Channel` and `Region` are **categorical codes**; treating them as continuous numerical distances would be misleading. Source: Cardoso (2013), UCI; dataset mirrored on Kaggle.

## Part 1 — Data quality assessment (10 points)

**Task 1.** Print dataset shape, column types, and descriptive statistics. Check that all eight required columns are present, and verify how many distinct category codes occur for `Channel` and `Region`.

In [ ]:
# TODO 1: Print df.shape, df.dtypes, df.describe() and category frequencies.
# Hints: df.dtypes, df['Channel'].value_counts(), df['Region'].value_counts()

**Task 2.** Count missing values, exact duplicate rows, and values that cannot represent nonnegative spending. Describe the policy you would follow if errors are found. Explain why unusually high legitimate spending should not be deleted automatically.

In [ ]:
# TODO 2: Validate missing values, duplicate rows, and impossible spending values.
# Hints: df.isna().sum(), df.duplicated().sum(), df[spend_cols].lt(0).sum()

**Reflection 1.** If two customers have exactly the same six spending values, is that sufficient evidence that one record is a duplicate? What information would you need to verify this?

## Part 2 — Exploratory data analysis (10 points)

**Task 3.** Visualize the distributions of all six spending variables. Compute and discuss skewness. Identify why a few very high-spending customers may influence Euclidean distances.

In [ ]:
# TODO 3: Plot 2x3 histograms and examine skewness.
# Hints: df[spend_cols].hist(), df[spend_cols].skew()

**Task 4.** Produce a correlation heatmap of the six spending variables and a boxplot of log-transformed spending. Which product categories appear to move together? Note that correlation does not imply causation.

In [ ]:
# TODO 4: Correlation heatmap and log1p boxplot for the six spending columns.
# Hints: sns.heatmap(), np.log1p(), sns.boxplot()

## Part 3 — Feature selection and scaling (15 points)

**Task 5.** Compare raw and `log1p`-transformed spending for at least two categories. Explain why `log1p` is defined when spending is zero, and why a log transform may be useful for strongly right-skewed spending.

In [ ]:
# TODO 5: Compare Fresh and Grocery before and after log1p transformation.

**Task 6.** Create the six-dimensional feature matrix using **only** the six spending categories. Apply `np.log1p()` and then `StandardScaler()`. Verify that the transformed matrix has the expected shape and that each feature has mean approximately 0 and standard deviation approximately 1. 

**Key point:** Standardization occurs **after** the log transform, not before it. For this descriptive clustering project, the whole dataset may be explored; it is not the same validation protocol as a supervised predictive task.

In [ ]:
# TODO 6: Create X_original, X_log, scaler, X_scaled.
# Hint: scaler.fit_transform(np.log1p(df[spend_cols]))

**Reflection 2.** Why is it inappropriate to include `Channel` and `Region` directly in Euclidean distances as raw integer codes? If you wanted to cluster mixed numerical and categorical variables, what would have to change?

## Part 4 — Linkage mechanics on a tiny example (10 points)

**Theory recap:** Agglomerative clustering starts with every observation in its own group. It repeatedly merges the two closest groups. “Closest” depends on the **linkage** definition:

- **Single linkage:** smallest pairwise distance between two groups.
- **Complete linkage:** largest pairwise distance.
- **Average linkage:** average of all cross-group pairwise distances.
- **Ward linkage:** chooses the merge with the smallest increase in within-cluster sum of squares (for Euclidean geometry); it is **not** simply the average distance between points.

**Task 7.** Suppose cluster A consists of points (0, 0), (0, 2), and cluster B consists of (3, 0), (3, 4). Compute the four cross-cluster Euclidean distances and single, complete, and average linkage distances. Compare the values; explain why these definitions might lead to different merges.

In [ ]:
# TODO 7: Calculate the distance matrix of A vs B.
# Hints: scipy.spatial.distance.cdist, distance_matrix.min()/max()/mean()

## Part 5 — Dendrograms and cutting the hierarchy (15 points)

**Task 8.** Fit **four SciPy hierarchies** (`single`, `complete`, `average`, and `ward`) using `X_scaled`. Plot their truncated dendrograms for readability. Explain the differences, including the chaining effect associated with single linkage. The vertical axis shows **merge height**, not a percentage or an accuracy score. Heights under different linkage methods are not necessarily on the same scale.

In [ ]:
# TODO 8: Create `trees` dictionary of linkage matrices and plot four dendrograms.
# Hints: linkage(X_scaled, method='ward'), dendrogram(..., truncate_mode='lastp', p=20)

**Task 9.** Plot the Ward dendrogram at a more detailed truncation level (`p=30`). Cut the Ward hierarchy to obtain **three clusters** using `cut_tree(..., n_clusters=3)` and display the cluster sizes. Explain how cutting a dendrogram higher or lower changes the number of clusters. 

**Technical note:** `cut_tree` assigns membership based on the SciPy hierarchy. Later we will also fit `AgglomerativeClustering` independently using scikit-learn.

In [ ]:
# TODO 9: Show Ward dendrogram and cut it to 3 clusters.
# Hints: dendrogram(trees['ward'], truncate_mode='lastp', p=30); cut_tree(..., n_clusters=3)

## Part 6 — Model selection and validation (15 points)

**Task 10.** Search across **2–8 clusters** for all four linkage methods using scikit-learn `AgglomerativeClustering`. For each candidate calculate the silhouette score, Calinski–Harabasz score, Davies–Bouldin score, and the smallest cluster size. Report a table sorted by silhouette score and a plot comparing silhouette values across `k` and linkage method.

Use the **same** `X_scaled` matrix to compare candidates. For silhouette, higher is generally better; for Davies–Bouldin, lower is generally better. These are internal measures and do not guarantee business usefulness.

In [ ]:
# TODO 10: Search methods x k in [2, 8], compute 3 indices and min cluster size.
# Hints: AgglomerativeClustering(n_clusters=k, linkage=method)
# and silhouette_score(X_scaled, labels)

**Task 11.** Select a final linkage method and `k`, **considering both silhouette and cluster sizes**. A simple operational rule for this class is to prefer candidates with at least 10 customers per segment; if none meets this threshold, explain how you made a defensible alternative choice. Fit the final agglomerative model and show the counts and proportions for all clusters.

**Important:** The highest silhouette can correspond to one huge cluster and one tiny group of outliers. Your final selection should be justified by the business task and inspected visually. The automatic rule in the instructor notebook is a starting point, **not** a claim of an objectively optimal segmentation.

In [ ]:
# TODO 11: Choose best_method and best_k using documented criteria.
# Create `final_model`, `labels`, and `cluster_sizes`.

## Part 7 — Visualization and customer profiles (15 points)

**Task 12.** Make a **two-component PCA scatterplot** of customers colored by final cluster. Add cluster counts. The model must still be fitted in the original **six-dimensional transformed space**; the PCA plot is a *projection for display* and can hide overlap.

In [ ]:
# TODO 12: Use PCA(n_components=2) and plot the cluster labels.

**Task 13.** Create a profile table of **median annual spending in original monetary units** by cluster, plus cluster counts. Then build a heatmap showing each cluster’s average standardized log-spend in the six dimensions. Why are medians often more informative than arithmetic means for skewed spending?

In [ ]:
# TODO 13: Profiles on raw scale and a relative feature heatmap on scaled log scale.
# Hints: df.assign(Cluster=labels).groupby('Cluster')[spend_cols].median()

**Task 14.** Compare cluster assignments against `Channel` and `Region` using **row-normalized cross-tabulations**. Write one sentence describing each segment based on the *observed purchasing variables*, without assuming that clustering has predicted a target label or caused customer behavior.

In [ ]:
# TODO 14: Compute cluster-by-Channel and cluster-by-Region percentages.
# Hints: pd.crosstab(segmented['Cluster'], segmented['Channel'], normalize='index')

## Part 8 — Sensitivity analysis and business action plan (10 points)

**Task 15.** Assess how results change when: (a) you omit log transformation, but still standardize; and (b) you use `RobustScaler` after `log1p`. Fit the **same final linkage and k** in both alternatives, calculate adjusted Rand index (ARI) relative to your final segmentation, and interpret the stability. As an *optional baseline*, fit K-means with the same `k` and compare silhouette on the main transformed data. 

ARI is **1** for identical partitions (up to label permutations), around 0 for chance-level agreement, and can be negative. It is not a measure of economic value.

In [ ]:
# TODO 15: Compare final labels to raw-standardized and log-RobustScaler partitions.
# Use adjusted_rand_score(labels, alt_labels).
# Optional: KMeans(n_clusters=best_k, random_state=42, n_init=10).

**Task 16.** Export the dataset with cluster assignments to `outputs/wholesale_customers_segmented.csv`, export the median profile table, and write a **one-page management memo** (200–350 words) answering:

1. What linkage method and `k` did you select, and why?
2. What distinguishes each segment in observed purchase patterns?
3. Which operational or marketing test would you run for each segment (e.g., product assortment, account management, targeted offers)?
4. What are two limitations of this dataset and your clustering method?

**Evaluation reminder:** This project creates business hypotheses, not experimentally verified treatment effects. Test recommendations before deploying them.

In [ ]:
# TODO 16: Export labeled observations and profile summary to CSV.
# Then write the 200–350-word memo in a separate Markdown or PDF file.

## Final discussion questions (short written answers)

1. Do agglomerative methods require fixing `k` before constructing the full tree? How is that different from deciding where to cut the dendrogram?
2. Why might single linkage create one large chained cluster while Ward produces more compact groups?
3. If two linkage methods produce similar silhouette scores but very different segment sizes, which result should be preferred? Justify.
4. Is the height of a Ward merger the same quantity as the average Euclidean distance between its two clusters?
5. What can `Channel` and `Region` tell you *after* clustering? Why should these categorical codes not automatically be included in numeric distance calculations?
6. How could you evaluate whether the proposed targeted offers genuinely increase revenue or customer satisfaction?

## Submission checklist

- [ ] Completed student `.ipynb` notebook, with all 16 tasks and their outputs.
- [ ] Linkage dendrograms, selected Ward/alternative dendrogram, PCA scatter, and feature profile heatmap.
- [ ] Model selection table and segment profile table; discussion of small clusters and sensitivity.
- [ ] CSV outputs, reproducible implementation (`random_state=42` for stochastic baseline), and 200–350-word management memo.
- [ ] Citation of the data source and acknowledgment of clustering limitations.

## References

- Cardoso, M. (2013). *Wholesale customers*. UCI Machine Learning Repository. DOI: https://doi.org/10.24432/C5030X (CC BY 4.0).
- Kaggle mirror: https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set
- SciPy: https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html
- Scikit-learn: https://scikit-learn.org/stable/modules/clustering.html#hierarchical-clustering
- Lecture: https://vudmvn.github.io/Data_Analysis_with_Python/#/lectures/part07-data-segmentation-clustering/part07-hierarchical-agglomerative-clustering-vn